In [1]:
import pandas as pd
import altair as alt

## Data

In [2]:
initial_data = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')
sample_50 = pd.read_csv('../data/sample/sample_50.csv')
sample_100 = pd.read_csv('../data/sample/sample_100.csv')

## Diagnostics

Bootstrapped sample contains similar KC coverage as original data.

In [3]:
def print_diagnostics(original: pd.DataFrame, bootstrapped: pd.DataFrame, label: str):
    print(f"\n{'='*62}")
    print(f"  {label}")
    print(f"{'='*62}")
 
    n_orig = original["student_id"].nunique()
    n_boot = bootstrapped["student_id"].nunique()
    print(f"  Students  : {n_orig:>5}  →  {n_boot}")
    print(f"  Total rows: {len(original):>5}  →  {len(bootstrapped)}")
 
    # percent_score value counts (discrete: 0, 50, 100)
    print("\n  percent_score distribution:")
    orig_pct = original["percent_score"].value_counts(normalize=True).sort_index()
    boot_pct = bootstrapped["percent_score"].value_counts(normalize=True).sort_index()
    print(f"    {'value':<8} {'original':>10} {'bootstrapped':>14}")
    for v in sorted(set(orig_pct.index) | set(boot_pct.index)):
        print(f"    {str(v):<8} {orig_pct.get(v, 0):>10.3f} {boot_pct.get(v, 0):>14.3f}")

 
    # avg attempts per student
    orig_avg = len(original) / n_orig
    boot_avg = len(bootstrapped) / n_boot
    print(f"\n  Avg attempts/student: {orig_avg:.1f}  →  {boot_avg:.1f}")

In [4]:
print_diagnostics(initial_data, sample_50, "Size of Sample : 50 students")


  Size of Sample : 50 students
  Students  :    25  →  50
  Total rows: 21808  →  43581

  percent_score distribution:
    value      original   bootstrapped
    0             0.439          0.458
    50            0.065          0.064
    100           0.496          0.478

  Avg attempts/student: 872.3  →  871.6


In [5]:
print_diagnostics(initial_data, sample_100, "Size of Sample : 100 students")


  Size of Sample : 100 students
  Students  :    25  →  100
  Total rows: 21808  →  87249

  percent_score distribution:
    value      original   bootstrapped
    0             0.439          0.448
    50            0.065          0.064
    100           0.496          0.488

  Avg attempts/student: 872.3  →  872.5


### Assignment Coverage

In [6]:
def coverage_chart(data, col, label):
    prop_df = (
        data[col]
        .value_counts(normalize=True)
        .reset_index()
        .rename(columns={"index": col, "proportion": "proportion"})
    )

    chart = alt.Chart(prop_df).mark_bar().encode(
        x=alt.X(f'{col}:N', title='Assignment'),   # :N = nominal/categorical
        y=alt.Y('proportion:Q', title='Number of Attempts', stack="normalize"), # :Q = quantitative
        tooltip=[f'{col}:N', 'count():Q']
    ).properties(
        title=label,
        width=400,
        height=300
    )
    return chart

In [7]:
coverage_chart(initial_data, 'assignment_id', '25 Student Sample (Original data)') | coverage_chart(sample_50, 'assignment_id', '50 Student Sample') | coverage_chart(sample_100,'assignment_id', '100 Student Sample')

alt.HConcatChart(...)

### KC Coverage

In [8]:
def kc_coverage_chart(data, label):
    data = pd.DataFrame(data["primary_kc_id"].value_counts().reset_index())
    chart = alt.Chart(data).mark_bar().encode(
    x=alt.X('count', title='Number of observations').bin(maxbins=30),
    y=alt.Y('count()', title='Number of KCs'),
    tooltip=['count()']  
    ).properties(
        title=label
    )
    return chart

In [9]:
kc_coverage_chart(initial_data, '25 Student Sample (Original data)') | kc_coverage_chart(sample_50,  '50 Student Sample') | kc_coverage_chart(sample_100, '100 Student Sample')

alt.HConcatChart(...)